In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q16/annotated/checkpoints/post_cell_2.pickle

trying: ['tpch_parent']
me:  0
trying: ['orig_output']
me:  4
trying: ['STORAGE_OPTS']
me:  0
trying: ['load_part']
me:  1
trying: ['part']
me:  1
trying: ['DATA_ROOT']
me:  0
trying: ['supplier']
me:  3
trying: ['load_supplier']
me:  3
trying: ['load_partsupp']
me:  2
trying: ['pd']
me:  0
trying: ['partsupp']


me:  2
Declaring variable tpch_parent
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable load_part
Declaring variable part
Declaring variable load_partsupp
Declaring variable partsupp
Declaring variable supplier
Declaring variable load_supplier
Declaring variable orig_output


In [4]:
%%RecordEvent
%%time
### cell 3 ###

# Optimized cell 3
# 1) Pre-compute the supplier keys to exclude via a single filter & drop_duplicates
sup_keys = (
    supplier['S_SUPPKEY']
    [supplier['S_COMMENT'].str.contains(r"Customer(\S|\s)*Complaints")]
    .drop_duplicates()
)

# 2) Filter parts by brand, type and size
part_filtered = (
    part[
        (part.P_BRAND != 'Brand#45')
        & ~part.P_TYPE.str.contains('^MEDIUM POLISHED')
        & part.P_SIZE.isin([49, 14, 23, 45, 19, 3, 36, 9])
    ]
    [['P_PARTKEY', 'P_BRAND', 'P_TYPE', 'P_SIZE']]
)

# 3) Narrow down partsupp
ps = partsupp[['PS_PARTKEY', 'PS_SUPPKEY']]

# 4) Inner-join parts → partsupp, then exclude unwanted suppliers via isin
merged = (
    part_filtered
    .merge(ps, left_on='P_PARTKEY', right_on='PS_PARTKEY')
    [['P_BRAND', 'P_TYPE', 'P_SIZE', 'PS_SUPPKEY']]
)
filtered = merged[~merged.PS_SUPPKEY.isin(sup_keys)]

# 5) Group, count unique suppliers, rename & sort in one pipeline
total = (
    filtered
    .groupby(['P_BRAND', 'P_TYPE', 'P_SIZE'], as_index=False, sort=False)
    ['PS_SUPPKEY']
    .nunique()
    .rename(columns={'PS_SUPPKEY': 'SUPPLIER_CNT'})
    .sort_values(
        by=['SUPPLIER_CNT', 'P_BRAND', 'P_TYPE', 'P_SIZE'],
        ascending=[False, True, True, True]
    )
)


CPU times: user 77.2 ms, sys: 20.3 ms, total: 97.5 ms
Wall time: 97.5 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q16/rewritten/o4_mini_high/checkpoints/post_cell_3_try_0.pickle

migration speed (bps): 802026265.8592318
---------------------------
variables to migrate:
filtered 48
part_filtered 48
STORAGE_OPTS 64
load_part 144
part 48
load_supplier 144
ps 48
partsupp 48
supplier 48
load_partsupp 144
pd 72
orig_output 16
sup_keys 48
tpch_parent 54
total 48
merged 48
DATA_ROOT 80
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q16/rewritten/o4_mini_high/checkpoints/post_cell_3_try_0.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['partsupp']
Future variables ['part']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['partsupp', 'part']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['sup_keys', 'merged', 'filtered', 'total', 'part_filtered', 'ps']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q16/opt_cell_exec_info_3_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q16/annotated/checkpoints/post_cell_3.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
